In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial
from pathlib import Path

#
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from Instruments.GX_271.gilson_ethernet import GilsonEthernet
from Instruments.GX_271.tray import Tray
from Instruments.GX_271.rack import Rack_209, Rack_3dp
from Core.logging import flow_logger as logger
from Instruments.GX_271.dim import DIM
from Instruments.WPI_Syr_pump.Pump import AL1000
from Instruments.Runze_valves.Runze62Valve import Runze62Valve

In [2]:
#--- Construct the tray ---
tray = Tray()

# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.101", admin_port=50185, tray=tray)

# --- Start logging (declare this run belongs to the experiment "Development") ---
logger.start_experiment("Development")

# --- Instantiate modules (racks, wash stations, etc.) (these know internal geometry) ---
rack1 = Rack_209()  
rack2 = Rack_3dp()
#DIM = DIM()

# --- Register modules on the tray with global offsets, previously handled by each module in the tray ---
tray.add_module(
    slot=1,
    name="rack1",
    module=rack1,
    x_offset=36.5,
    y_offset=7.6,
    x_min=27,
    x_max=127,
    y_min=2,
    y_max=292
)

tray.add_module(
    slot=2,
    name="rack2",
    module=rack2,
    x_offset=201,
    y_offset=39,
    x_min=157,
    x_max=247,
    y_min=2,
    y_max=292
) 

In [3]:
# --- Instantiate serial object for AL1000 pump ----
ser = serial.Serial("COM2", 9600, timeout=0.5)
pump = AL1000(ser, device_address="@00", sleep_time=0.5)

pump.connect()

Device is connected


In [4]:
syringe_dia = 4.606    # mm
rate = 0.5            # mL/min
volume = 200           # uL
direction = "WDR"     # WDR = withdraw, INF = Infuse

pump.prepare(dia=syringe_dia, rate=rate, volume=volume, direction=direction)


--- Preparing pump (Phase 1) ---
Sent: @00*PHN1
Sent: @00*DIA4.606
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*RAT C 0.5 MM
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*VOL200
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*DIRWDR
Raw reply: 00S
Parsed reply: Acknowledged = True
--- Preparation complete ---



In [22]:
g.go_to_vial("rack1", 1)

[DEBUG] ensure_z_safe: current_z=93.0 is already >= target_z=93


(np.float64(201.0), np.float64(39.0), 1)

In [17]:
pump.start()

Sent: @00*RUN 1


'00I'

In [2]:
# --- Instantiate the runze valve --- 

runze = Runze62Valve(
    port="COM7",
    baudrate=9600,
    timeout=1
)

runze.connect()

#valve.go_to_pos(1)
#valve.go_to_pos(2)
#valve.toggle()


Connected to Runze valve on COM7
Valve moved to position 1


In [24]:
g.home()

Z below safe limit (96.00 < 100.00) — raising first.
All axes homed successfully and Z is within safe limits


In [3]:
runze.read_pos()

1

In [9]:
g.go_to_vial("rack1", 1)

[DEBUG] ensure_z_safe: current_z=93.0 is already >= target_z=45.0
[DEBUG] ensure_z_safe: current_z=93.0 is already >= target_z=93


(np.float64(36.5), np.float64(7.6))

In [5]:
time.sleep(5)
g.go_into_vial("rack2", 1)
time.sleep(2)
pump.start()

[DEBUG] ensure_z_safe: current_z=93.0 is already >= target_z=93
[DEBUG] ensure_z_safe: current_z=93.0 is already >= target_z=93
Sent: @00*RUN 1


'00W'

In [6]:
g.go_to_vial("rack1", 1)
g.move_z(45)

syringe_dia = 4.606    # mm
rate = 0.5            # mL/min
volume = 200           # uL
direction = "INF"     # WDR = withdraw, INF = Infuse

pump.prepare(dia=syringe_dia, rate=rate, volume=volume, direction=direction)

pump.start()

[DEBUG] ensure_z_safe: current_z=93.0 is already >= target_z=93

--- Preparing pump (Phase 1) ---
Sent: @00*PHN1
Sent: @00*DIA4.606
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*RAT C 0.5 MM
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*VOL200
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*DIRINF
Raw reply: 00S
Parsed reply: Acknowledged = True
--- Preparation complete ---

Sent: @00*RUN 1


'00I'

In [7]:
g.move_z(80, allow_in_vial=True)

'Moved Z to 80. Result: No valid response received.'

In [14]:
g.send_command("Get XY Position")

'Get XY Position: 35.6, 8.1'

In [3]:
g.move_z(70)

'Moved Z to 70. Result: No valid response received.'

In [34]:
g.move_y(7.6)

[DEBUG] ensure_z_safe: current_z=45.0 is already >= target_z=45.0


'Moved Y to 7.6. Result: Move Y: 2, Success'

In [21]:
g.go_into_vial("rack2", 1,)

[DEBUG] ensure_z_safe: current_z=95 is already >= target_z=93
[DEBUG] ensure_z_safe: current_z=95 is already >= target_z=45.0


(np.float64(201.0), np.float64(39.0), 1)

In [37]:
print(g.current_x)

36.5


In [38]:
print(g.current_y)

7.6


In [10]:
g.go_to_vial("rack2", 1)

[DEBUG] ensure_z_safe: current_z=93 is already >= target_z=45.0


(np.float64(201.0), np.float64(39.0))

In [11]:
print(g.current_module)

rack1
